# Portfolio Positions

This notebook builds the current position for each asset and account pair from the stored transactions. Purchases increase quantity and invested capital, sales decrease them, and custody transfers move quantity and invested capital between accounts. Dividends, interest, and income are excluded. The computed positions are saved under `data/store/computed_positions`.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.core.repositories.position_repository import PositionRepository
from src.core.repositories.transaction_repository import TransactionRepository
from src.core.services.position_build import PositionBuildService

In [ ]:
transaction_repository = TransactionRepository(
    project_root / 'data/store/transactions/transactions.csv'
)
position_repository = PositionRepository(
    project_root / 'data/store/computed_positions/positions.csv'
)

position_service = PositionBuildService(
    transaction_repository=transaction_repository,
    position_repository=position_repository,
)
positions = position_service.build_and_save()

print(f'Computed and saved {len(positions)} positions.')

In [ ]:
import pandas as pd
from IPython.display import display

position_table = pd.DataFrame([
    {
        'account_id': position.account_id,
        'asset_id': position.asset_id,
        'quantity': position.quantity,
        'invested_capital': position.invested_capital,
    }
    for position in positions
    if position.quantity != 0
]).sort_values(['account_id', 'asset_id']).reset_index(drop=True)

display(position_table)

In [ ]:
# Confirm that the persisted file can be read back as typed Position objects.
saved_positions = position_repository.all()
assert saved_positions == positions
print(f'Verified {len(saved_positions)} persisted positions.')